# Actividad | Dataset - Notebook | Individual

- Alumno: Emilio Alejandro Martínez López (A01796936)
- Fecha: 4 Octubre 2026
- Curso: TC5061 - Operaciones de Aprendizaje Automático

## Exploración de préstamos históricos

Este notebook describe la muestra reproducible; no prepara datos de entrenamiento, ajusta modelos ni registra experimentos. Para entrenar y generar evidencia de evaluación, ejecutar `python train.py` desde la raíz del proyecto.

Con el entorno del README instalado, seleccionar su intérprete como kernel y ejecutar todas las celdas en orden. Las rutas predeterminadas se resuelven desde el paquete instalado, no desde el directorio del notebook. Para otros datos, editar únicamente la celda de configuración marcada `parameters`.


## Contexto del dataset

- Dataset: [Credit Scoring | Lending Club Loan Data Cleared](https://www.kaggle.com/datasets/db0boy/lending-club-loan-data-cleared/)
- Descripción: El dataset contiene información histórica de préstamos otorgados a través de LendingClub y está orientado principalmente al análisis y predicción del riesgo crediticio.
- Estado de pago observado en la fuente: 0 = préstamo pagado, 1 = préstamo impagado. Se analiza su asociación con información disponible durante la originación del préstamo, sin atribuir causalidad.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from credit_risk import TrainingConfig, load_historical_loan_sample


In [ ]:
config = TrainingConfig()


## 1. Muestra reproducible y calidad

La API pública valida los CSV, selecciona hasta 100,000 registros de forma estratificada con semilla 42 por defecto y audita todas las columnas originales. Reduce copias exactas a un registro y excluye todos los registros con estados de pago contradictorios. La auditoría solo cubre la muestra seleccionada, no los registros fuente fuera de ella; las exclusiones pueden alterar las proporciones.

Los valores por registro permanecen en formato fuente, sin imputación, codificación ni limpieza de entrenamiento. No se divide entrenamiento/prueba: las tablas siguientes describen toda la muestra retenida.


In [ ]:
muestra = load_historical_loan_sample(config)
prestamos_historicos = muestra.features.copy()
prestamos_historicos['estado_de_pago'] = muestra.payment_status
calidad = muestra.data_quality

print(f'Semilla: {config.seed}')
print(f'Registros fuente: {calidad.source_rows:,}')
print(f'Registros seleccionados: {calidad.sampled_rows:,}')
print(f'Copias exactas excluidas: {calidad.duplicate_rows_removed:,}')
print(f'Registros contradictorios excluidos: {calidad.contradictory_rows_removed:,}')
print(f'Préstamos históricos retenidos: {calidad.retained_rows:,}')


### Estructura y diccionario de datos

In [ ]:
display(pd.DataFrame({'variable': muestra.features.columns, 'tipo': muestra.features.dtypes.astype(str).to_numpy()}))


#### Significado de las variables

- `funded_amnt` (numérica): Monto financiado (USD).
- `interest_rate` (numérica): Tasa de interés anual (%).
- `grade` (categórica): Subgrado crediticio del proveedor (A1–G5).
- `emp_length` (texto): Antigüedad laboral en formato fuente; categorías abiertas.
- `home_ownership_status` (categórica): Situación de vivienda.
- `annual_income` (numérica): Ingreso anual declarado (USD).
- `verification_status` (categórica): Estado de verificación del ingreso.
- `loan_purpose` (categórica): Propósito del préstamo.
- `dept_paym_income_ratio` (numérica): Relación pagos de deuda / ingreso (DTI, %).
- `num_30+_delinq_in_2yrs` (numérica): Morosidades de 30 o más días en los últimos dos años.
- `num_inq_in_6mths` (numérica): Consultas crediticias en los últimos seis meses.
- `num_open_credit_lines` (numérica): Número de líneas de crédito abiertas.
- `num_derogatory_pub_rec` (numérica): Número de registros públicos negativos.
- `total_credit_revolving_bal` (numérica): Saldo total de crédito revolvente (USD).
- `used_credit_share` (numérica): Utilización de crédito revolvente (%).
- `tot_num_credit_lines` (numérica): Total histórico de líneas de crédito.
- `loan_term_months` (numérica): Plazo contractual en meses.
- `remaining_princ_for_tot_amnt_fund` (numérica): Principal financiado aún pendiente (USD).
- `paym_rec_for_tot_amnt_fund` (numérica): Pagos recibidos sobre monto financiado (USD).
- `princ_rec` (numérica): Principal recibido (USD).
- `interest_rec` (numérica): Intereses recibidos (USD).
- `late_fees_rec` (numérica): Cargos por mora recibidos (USD).
- `monthly_payment` (numérica): Cuota mensual (USD); redundante con monto, tasa y plazo.
- `emp_title` (texto): Título o puesto laboral; alta cardinalidad.
- `addr_state` (categórica): Estado de residencia; ubicación.
- `mths_since_last_delinq` (numérica): Meses desde última morosidad; faltante de significado no verificado.
- `initial_list_status` (categórica): Estado inicial de listado; código del proveedor.
- `num_open_trades_in_6mths` (numérica): Líneas abiertas en últimos seis meses.
- `num_installment_acc_op_in_12mths` (numérica): Cuentas a plazos abiertas en últimos doce meses.
- `num_installment_acc_op_in_24mths` (numérica): Cuentas a plazos abiertas en últimos veinticuatro meses.
- `mths_since_last_installment_acc_op` (numérica): Meses desde apertura de última cuenta a plazos.
- `num_rev_trades_op_in_12mths` (numérica): Líneas revolventes abiertas en últimos doce meses.
- `num_rev_trades_op_in_24mths` (numérica): Líneas revolventes abiertas en últimos veinticuatro meses.
- `max_bal_owed` (numérica): Saldo máximo adeudado; definición exacta por confirmar.
- `bal_to_cred_lim` (numérica): Relación saldo / límite de crédito; escala por confirmar.
- `num_inq` (numérica): Consultas crediticias; ventana por confirmar.
- `num_inq_in_12mths` (numérica): Consultas crediticias en últimos doce meses.
- `mths_since_recent_bankcard_delinq` (numérica): Meses desde morosidad reciente en tarjeta bancaria.
- `mths_since_recent_revol_delinq` (numérica): Meses desde morosidad reciente de crédito revolvente.
- `disbursement_method` (categórica): Método de desembolso; se excluye por disponibilidad temporal incierta.
- `issue_date_month` (categórica): Mes de emisión; contexto temporal.
- `issue_date_year` (numérica): Año de emisión; contexto temporal.
- `region_code` (categórica): Código de región; ubicación redundante.
- `earliest_cr_line_month` (categórica): Mes de primera línea de crédito.
- `earliest_cr_line_year` (numérica): Año de primera línea de crédito.

El diccionario conserva el contexto de las columnas fuente, incluso las opcionales. Los campos de pagos recibidos, principal pendiente, intereses y cargos por mora son posteriores a la originación y no son predictores admisibles. El subgrado crediticio incorpora la evaluación previa del proveedor: usarlo no constituye una evaluación crediticia independiente.


## 2. Análisis descriptivo

### Resumen financiero y antigüedad laboral

Seleccionamos estas cinco variables por su relevancia financiera: representan monto, tasa, ingresos, carga de deuda y saldo revolvente.

Este resumen facilita identificar distribuciones y posibles valores extremos, no limita los predictores que se utilizarán posteriormente en el modelo.

In [ ]:
prestamos_historicos[['funded_amnt', 'interest_rate', 'annual_income',
                    'dept_paym_income_ratio', 'total_credit_revolving_bal']].describe().T

Se cuentan las categorías de antigüedad laboral, incluidos los faltantes, sin convertirlas ni imputarlas.

In [ ]:
prestamos_historicos['emp_length'].value_counts(dropna=False).to_frame('registros')

### Estado de pago y valores faltantes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
counts = prestamos_historicos['estado_de_pago'].value_counts().reindex([0, 1], fill_value=0)

axes[0].bar(
    ['0: préstamo pagado', '1: préstamo impagado'],
    counts.to_numpy(),
    color=['#4477AA', '#CC6677']
)

axes[0].set(
    title='Distribución de préstamos por estado de pago',
    ylabel='Préstamos'
)

axes[0].set_ylim(0, counts.max() * 1.15)

for i, count in enumerate(counts):
    axes[0].text(
        i,
        count,
        f'{count:,} ({count / counts.sum():.1%})',
        ha='center',
        va='bottom',
        fontsize=9
    )

missing = (
    prestamos_historicos.isna()
    .mean()
    .mul(100)
    .sort_values()
    .tail(14)
)

missing.plot.barh(
    ax=axes[1],
    color='#4477AA'
)

axes[1].set(
    title='Variables con mayor proporción de faltantes',
    xlabel='Faltantes (%)',
    ylabel=''
)

axes[1].tick_params(axis='both', labelsize=9)

plt.tight_layout()
plt.show()


Las proporciones mostradas corresponden a esta ejecución, no al antiguo subconjunto de entrenamiento. El desbalance entre préstamos pagados e impagados requiere cautela al interpretar resultados de modelos; la evaluación repetible corresponde al script, no a este notebook.

Un faltante puede representar ausencia del evento o información desconocida: su significado debe verificarse por campo, no suponerse a partir del porcentaje. Aquí se muestra su frecuencia sin modificar los registros.


### Distribuciones y posibles valores extremos

Los gráficos de ingreso y saldo usan log1p para visualizar las colas, esto permite ver mejor la distribución de los datos y detectar posibles valores extremos que podrían influir en el modelo.

El logaritmo `log1p` se usa solo en las vistas de ingreso y saldo no negativos; no cambia la muestra ni constituye limpieza de entrenamiento. Los ejes señalan esa escala.

In [ ]:
columnas_financieras = ['funded_amnt', 'interest_rate', 'annual_income', 'total_credit_revolving_bal']

fig, axes = plt.subplots(2, 4, figsize=(15, 6))

for i, columna in enumerate(columnas_financieras):
    valores = prestamos_historicos[columna].dropna()
    if columna in ['annual_income', 'total_credit_revolving_bal']:
        valores = np.log1p(valores[valores >= 0])
    sns.histplot(valores, bins=40, ax=axes[0, i])
    sns.boxplot(x=valores, ax=axes[1, i])
    escala = f'{columna} (log1p)' if columna in ['annual_income', 'total_credit_revolving_bal'] else columna
    axes[0, i].set_title(escala)
    axes[0, i].set_xlabel(escala)
    axes[1, i].set_xlabel(escala)

plt.tight_layout()
plt.show()

In [ ]:
q1 = prestamos_historicos[columnas_financieras].quantile(0.25)
q3 = prestamos_historicos[columnas_financieras].quantile(0.75)
iqr = q3 - q1

extremos = (prestamos_historicos[columnas_financieras] < q1 - 1.5 * iqr) | (prestamos_historicos[columnas_financieras] > q3 + 1.5 * iqr)

extremos.sum().rename('extremos_IQR_muestra').to_frame()

In [ ]:
print("Ingreso mediano en $USD:", prestamos_historicos.annual_income.median())
print("Media en $USD:", prestamos_historicos.annual_income.mean())

Los extremos por IQR son una descripción, no una regla automática para excluir préstamos. Comparar media y mediana ayuda a reconocer asimetría y la influencia de valores extremos; las cifras se recalculan con la muestra actual.

### Relaciones entre variables

In [ ]:
columnas_correlacion = ['funded_amnt', 'interest_rate', 'annual_income', 'dept_paym_income_ratio',
                       'total_credit_revolving_bal', 'used_credit_share',
                       'num_open_credit_lines', 'tot_num_credit_lines']

correlaciones = prestamos_historicos[columnas_correlacion].corr(method='spearman')

plt.figure(figsize=(9, 7))
sns.heatmap(correlaciones, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1)
plt.title('Correlaciones Spearman')
plt.tight_layout()
plt.show()

La matriz muestra asociaciones monotónicas en la muestra actual. Correlación no implica causalidad: relaciones entre saldos, líneas de crédito o tasa de interés pueden reflejar características compartidas y decisiones previas del proveedor.

### Subgrado crediticio, plazo y situación de vivienda frente al estado de pago

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, columna, titulo in zip(axes, ['grade', 'loan_term_months', 'home_ownership_status'], ['Subgrado crediticio', 'Plazo (meses)', 'Situación de vivienda']):
    tasas = prestamos_historicos.groupby(columna)['estado_de_pago'].agg(registros='size', impago='mean')
    (tasas['impago'] * 100).plot.bar(ax=ax)
    ax.set(title=titulo, ylabel='Impago observado (%)', xlabel='')
    display(tasas)

plt.tight_layout()
plt.show()

Las tablas incluyen el número de registros junto a la proporción observada de préstamos impagados. Comparar categorías con pocos registros puede resultar engañoso; las diferencias no prueban efectos causales. El subgrado crediticio ya expresa una evaluación de riesgo del proveedor durante la originación.

Las tasas se recalculan en cada ejecución: no se conservan como conclusiones las cifras del antiguo subconjunto de entrenamiento. Este análisis descriptivo no estima desempeño sobre préstamos nuevos.


## 3. Reflexión

Separar exploración y entrenamiento evita mantener dos implementaciones de validación, muestreo y auditoría. La configuración y la muestra compartidas permiten comprender el origen de las cifras, mientras el script concentra las transformaciones, el ajuste y la evaluación. No se registra ningún experimento desde este notebook.

En mi trabajo con dashboards ejecutivos y operativos de ciberseguridad, estos principios también son útiles: validar campos obligatorios, registros duplicados y fechas de actualización evita presentar información incompleta como si reflejara el estado actual. Documentar las definiciones y versionar los procesos ayuda a explicar por qué cambió una cifra. La integridad, trazabilidad y monitoreo de los datos son necesarios tanto para modelos como para dashboards confiables.


## Referencias

- db0boy. (s. f.). *Lending Club loan data cleared* [Conjunto de datos]. Kaggle. [https://www.kaggle.com/datasets/db0boy/lending-club-loan-data-cleared/data](https://www.kaggle.com/datasets/db0boy/lending-club-loan-data-cleared/data)

- Géron, A. (2019). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow: Concepts, tools, and techniques to build intelligent systems* (2.ª ed.). O’Reilly Media.

- OpenAI. (2026). *ChatGPT (GPT-5.6 Sol)* [Modelo de lenguaje grande], utilizado para revisión, corrección, debugging y refactorización de código. [https://chatgpt.com/](https://chatgpt.com/)

- Treveil, M., Omont, N., Stenac, C., Lefevre, K., Phan, D., Zentici, J., Lavoillotte, A., Miyazaki, M., & Heidmann, L. (2020). *Introducing MLOps: How to scale machine learning in the enterprise*. O’Reilly Media.